# *C. elegans* pipeline — Fig. 3 / Supp. Figs. 2, 3, 5, 6

This notebook applies the multi-timescale transfer-operator pipeline of
**Kaur, Jain, & Berman (2026)**, *"Using timescale as a state coordinate
reveals the metastable geometry of behavior,"* to freely-moving
*C. elegans* posture, recovering the two metastable basins — *pirouette*
and *run*.

It runs end-to-end from the cached artifacts shipped in `data/worms/`
(cluster sequences, eigenvectors, G-PCCA memberships, the UMAP enrichment
grid, and per-cluster kinematics).  The raw → wavelet → PCA → embed →
cluster *upstream* is documented but gated behind `RECOMPUTE_UPSTREAM`,
since the wavelet-PC intermediates are large and are not shipped.

In [ ]:
# --- Colab / local setup ---
# If you're running on Colab, uncomment the next lines to install the pip
# dependencies that aren't in the default Colab image and to clone this repo
# (for the helper modules + cached data in data/worms/).
#
# !pip install pygpcca powerlaw umap-learn
# !git clone https://github.com/bermanlabemory/slowmode.git
# %cd slowmode

import os, sys, pickle, time
import numpy as np
import matplotlib.pyplot as plt

# Ensure slowmode/ is on the path.
sys.path.insert(0, os.path.abspath('.'))

import pipeline as pp
import gpcca_utils as gu
import figures as fg

fg.setup_style()
os.makedirs('outputs', exist_ok=True)

In [ ]:
# --- imports specific to this notebook ---
from sklearn.decomposition import PCA
import warnings; warnings.filterwarnings('ignore', category=UserWarning)


## 1. Load the worm wavelet-PC and eigenworm data

`worms_wlets_pca.pkl` is a list of 12 worms; for each worm, a list of 5
contiguous valid segments (gaps where tracking failed cut the recording);
each segment is a $(T_\mathrm{seg}, 5)$ array of wavelet PCA projections.

`worms_eigenworms.pkl` is a $(386402, 5)$ MaskedArray of eigenworm
coefficients (concatenated time-domain), used for the Panel A power
spectrum and Panel E $\bar{\omega}$ computation.

`worms_state_partitions.pkl` is a $\{worm \to [\text{int arrays}]\}$
dict of cached cluster sequences from $k$-means at $N = 250$.


In [ ]:
# Load the shipped, precomputed worm data.  Everything needed to reproduce
# Fig. 3 and Supp. Figs. S2/S3/S5/S6 ships in data/worms/.
DATA_DIR = 'data/worms'

# Cluster sequences: one integer label per frame, per worm (N = 250 clusters).
with open(f'{DATA_DIR}/states_worms.pkl', 'rb') as f:
    states_flat = {w: np.asarray(s, dtype=int)
                   for w, s in pickle.load(f).items()}
n_worms = len(states_flat)

# Eigenworm coefficients (Stephens 2008 / Broekmans 2016): used for the PSD
# and the (omega, |theta|) kinematic plane.
with open(f'{DATA_DIR}/all_valid_segments_worms.pkl', 'rb') as f:
    eigenworms = np.ma.compress_rows(pickle.load(f))

print(f'{n_worms} worms; pooled frames = '
      f'{sum(len(s) for s in states_flat.values())}')
print(f'eigenworms shape = {eigenworms.shape}')

# --- Upstream (raw -> wavelet -> PCA -> embed -> cluster) -----------------
# The wavelet-PC intermediates are large and are NOT shipped.  To rebuild the
# cluster sequences from the raw eigenworm series, set RECOMPUTE_UPSTREAM =
# True and supply the per-segment eigenworm coefficients; the recipe is
#   pp.morlet_wavelet_amplitudes -> pp.pca_with_shuffle_threshold
#   -> pp.delay_embed -> pp.kmeans_partition
# (see Methods, "Transfer operator computations").  By default we use the
# shipped cluster sequences loaded above.
RECOMPUTE_UPSTREAM = False

## 2. Concatenate cluster sequences and build $T(\tau)$

We pool cluster sequences across worms and segments to build a single
$T(\tau)$ at $\tau = 3$ s (48 frames at 16 Hz).


In [ ]:
fs = 16.0
tau_seconds = 3.0
lag = int(tau_seconds * fs)            # 48 frames

# Keep the per-worm sequences as a LIST and pass the list to
# make_transition_matrix, so transitions are counted within each worm only
# (never across the splice between worms).  all_states is the concatenation,
# used only for per-frame indexing further down.
states_list = [states_flat[w] for w in sorted(states_flat)]
all_states = np.concatenate(states_list)
N = int(all_states.max()) + 1
print(f'pooled states: T = {len(all_states)}    N = {N}    lag = {lag} frames')

T_multi = pp.make_transition_matrix(states_list, lag=lag, n_states=N)
pi_multi = pp.stationary_distribution(T_multi)
evals_multi, evecs_multi = pp.leading_eigvecs(T_multi, k=10)
print(f'leading |lambda_k| = {np.abs(evals_multi)[:5].round(4)}')
print(f'spectral gap |lambda_2|/|lambda_3| = '
      f'{np.abs(evals_multi[0]) / np.abs(evals_multi[1]):.3f}')

## 3. G-PCCA at $M = 2$

The largest spectral gap selects $M = 2$ (run vs pirouette).


In [ ]:
out = gu.run_gpcca(T_multi, M=2, eta=pi_multi)
print(f'crispness     = {out["crispness"]:.3f}')
print(f'basin counts  = {out["basin_counts"]}')
print(f'pi per basin  = {out["pi_basin"].round(3)}')

# G-PCCA returns basins up to a column permutation.  For consistency with the
# published figures we adopt the cached memberships (same clusters, fixed
# labeling) for the downstream panels.
gp_cached = np.load(f'{DATA_DIR}/gpcca_worms_M2_tau3s.npz')
chi = gp_cached['chi']

### 3a. Anchoring the basin labels: the Costa $(\bar\omega, |\theta|)$ plane

G-PCCA returns two basins, but with no biological labels — it just picks the partition with the largest crispness.  To turn "Basin 1 / Basin 2" into "*Pirouette* / *Run*", we project each cluster onto the canonical kinematic plane used by Costa et al. (2024):

- $\bar\omega$ — the cluster-mean phase velocity (positive = forward propagating body wave, negative = reversal).
- $|\theta|$ — the cluster-mean curvature magnitude (large in sharp turns and ventral coiling, small in straight runs).

Reversals (negative $\bar\omega$) plus sharp turns (large $|\theta|$) define the classical *pirouette*; forward locomotion (positive $\bar\omega$, low $|\theta|$) defines the *run*.  Compute these per cluster, look at where each basin's $\pi$-weighted centroid sits in the plane, and the label assignment falls out automatically.

This is a **post-hoc** labelling — the partition itself is identified by the operator alone, with no behavioral input.  The Costa plane only enters when we need a name to call each basin.

In [ ]:
# Per-cluster (omega_bar, |theta|) for the M = 2 partition, shipped in
# data/worms/worms_costa_per_cluster.npz.
costa = np.load(f'{DATA_DIR}/worms_costa_per_cluster.npz')
omega_per_cluster = costa['om_mean']            # (N_clusters,)
theta_per_cluster = costa['gp_mean']            # (N_clusters,) mean |theta|

# Soft-membership-weighted basin centroids in the (omega, |theta|) plane.
basin_centroids = []
fin = np.isfinite(omega_per_cluster) & np.isfinite(theta_per_cluster)
for j in range(2):
    w = (chi[:, j] * pi_multi)[fin]
    om_c = np.average(omega_per_cluster[fin], weights=w)
    th_c = np.average(theta_per_cluster[fin], weights=w)
    basin_centroids.append((om_c, th_c))
    print(f'Basin {j+1}: <omega_bar> = {om_c:+.2f} body wave/s,  '
          f'<|theta|> = {th_c:.2f} rad')

pirouette_basin = int(np.argmin([c[0] for c in basin_centroids]))
run_basin = 1 - pirouette_basin
print(f'\n=> Basin {pirouette_basin+1} = Pirouette   (lower omega_bar, larger |theta|)')
print(f'=> Basin {run_basin+1} = Run         (higher omega_bar, smaller |theta|)')

## Fig. 3A — PSD of the 5 eigenworm coefficients

Power spectral densities of the five eigenworm coefficients across all
worms, with the wavelet-band cutoffs (0.1 and 8 Hz) annotated.


In [ ]:
from scipy.signal import welch
ew = np.asarray(eigenworms)
fig, ax = plt.subplots(figsize=(4.4, 3.0))
for k in range(5):
    f, Pxx = welch(ew[:, k], fs=fs, nperseg=4096)
    ax.loglog(f, Pxx, lw=1.0, alpha=0.85, label=f'eigenworm {k+1}')
ax.axvline(0.1, ls='--', color='r', lw=0.7)
ax.axvline(8.0, ls='--', color='r', lw=0.7)
ax.set_xlabel('frequency (Hz)'); ax.set_ylabel('PSD')
ax.set_xlim(1e-2, fs / 2)
ax.legend()
plt.tight_layout()
fg.save_panel(fig, 'fig3A_eigenworm_psd')
plt.show()


## Fig. 3B — Eigenvalue spectrum


In [ ]:
fig, ax = plt.subplots(figsize=(4.0, 2.8))
fg.plot_eigenvalue_dots(ax, evals_multi[:10], M=2)
plt.tight_layout()
fg.save_panel(fig, 'fig3B_eigenvalue_spectrum')
plt.show()


## Fig. 3C / 3D — UMAP-embedded basin enrichment maps

We compute a 2-D UMAP of the per-frame delay-embedded wavelet-PC vectors,
and render the per-bin mean of $\chi_1$ (Fig. 3C: pirouette) and $\chi_2$
(Fig. 3D: run) as RdBu_r heatmaps.

UMAP fitting on ~400 k frames takes a few minutes; for smoothness we sample
60 k frames stratified across worms.


In [ ]:
# Fig. 3C/D use the canonical UMAP embedding of the delay-embedded wavelet-PC
# space.  Refitting the 2D embedding is expensive and needs the (un-shipped)
# wavelet PCs, so by default we load the shipped per-basin enrichment grid.
if RECOMPUTE_UPSTREAM:
    raise NotImplementedError(
        'Refit UMAP from wavelet PCs here (umap.UMAP on the delay-embedded '
        'wavelet-PC space); see Methods.')
umap_grid = np.load(f'{DATA_DIR}/worms_umap_canonical_full.npz')
gx, gy = umap_grid['x_edges'], umap_grid['y_edges']
dev = umap_grid['dev']     # (2, ny, nx): per-basin chi enrichment above the mean
print(f'UMAP enrichment grid: dev shape = {dev.shape}')

In [ ]:
# Fig. 3C / 3D: per-basin chi enrichment on the canonical UMAP embedding.
vmax = float(np.nanpercentile(np.abs(dev), 99))
panel_info = [('C', 'Pirouette', '#E69F00'), ('D', 'Run', '#56B4E9')]
for j, (letter, name, color) in enumerate(panel_info):
    fig, ax = plt.subplots(figsize=(4.4, 4.4))
    im = ax.pcolormesh(gx, gy, dev[j], cmap='RdBu_r', vmin=-vmax, vmax=vmax,
                       shading='auto', rasterized=True)
    fig.colorbar(im, ax=ax, fraction=0.045, pad=0.02,
                 label=fr'$\chi_{j+1}$ enrichment')
    ax.set_aspect('equal'); ax.set_xticks([]); ax.set_yticks([])
    for sp in ax.spines.values():
        sp.set_visible(False)
    ax.set_title(f'Basin {j+1} ({name}) enrichment on UMAP', color=color)
    fg.save_panel(fig, f'fig3{letter}_chi{j+1}_umap')
    plt.show()

## Fig. 3E — $(\bar{\omega}, |\theta|)$ behavioral plane

Per-cluster scatter on the canonical Costa et al. (2024) behavioral plane
of cluster-mean phase velocity $\bar{\omega}$ and cluster-mean
tangent-angle magnitude $\langle|\theta|\rangle$, colored by basin-1
membership $\chi_1$.  Marker size is proportional to $\pi$.

We compute $\omega(t)$ as the time-derivative of the phase angle in the
(eigenworm-1, eigenworm-2) plane, and $|\theta|$ as the L2 norm of the
five eigenworm coefficients.


In [ ]:
# Fig. 3E: per-cluster (omega_bar, |theta|) kinematic plane, coloured by chi_1.
# Uses the shipped per-cluster kinematics (loaded above as omega/theta_per_cluster).
ok = np.isfinite(omega_per_cluster) & np.isfinite(theta_per_cluster)
fig, ax = plt.subplots(figsize=(4.8, 3.8))
sizes = 18 + 280 * pi_multi[ok] / pi_multi[ok].max()
sc = ax.scatter(omega_per_cluster[ok], theta_per_cluster[ok], c=chi[ok, 0],
                cmap='RdBu_r', vmin=0, vmax=1, s=sizes, edgecolor='0.25',
                linewidth=0.3, alpha=0.85)
ax.axvline(0, color='0.5', lw=0.5, ls='--')
ax.set_xlabel(r'cluster $\bar\omega$ (body wave/s)')
ax.set_ylabel(r'cluster $\langle|\theta|\rangle$ (rad)')
fig.colorbar(sc, ax=ax, fraction=0.045, pad=0.02, label=r'$\chi_1$')
plt.tight_layout()
fg.save_panel(fig, 'fig3E_omega_theta_plane')
plt.show()

## Fig. S2 — operator diagnostics

The operator-diagnostic supplementary panels: Cao's $E_1(d)$, the
entropy-gap criterion, eigenvalue spectra (multi vs fixed), and
basin-level non-Markovianity ($r_k(\tau)$ vs lag).

These cells run on the full pooled wavelet-PC array.  Reduce `N_values` or
`max_d` for a faster demo.


In [ ]:
# Fig. S2A: Cao's E_1(d) on the worm wavelet PCs.  Needs the (un-shipped)
# wavelet PCs; the published panel is regenerated by figures/supp_figure_2.py
# from data/worms/worms_supp_data.npz.
if RECOMPUTE_UPSTREAM:
    E1 = pp.cao_e1(all_pcs, max_d=15, tau=1, n_samples=20000, seed=0)
    fig, ax = plt.subplots(figsize=(4.0, 2.8))
    ax.plot(np.arange(2, len(E1) + 2), E1, 'o-', color='C0')
    ax.axhline(1.0, ls=':', color='0.6')
    ax.set_xlabel(r'$d$'); ax.set_ylabel(r'$E_1(d)$')
    plt.tight_layout(); fg.save_panel(fig, 'figS2A_cao_E1'); plt.show()
else:
    print('Fig. S2A (Cao E_1) needs the wavelet PCs; see figures/supp_figure_2.py')

In [ ]:
# Fig. S2B: entropy-gap Delta H(N).  Needs the (un-shipped) wavelet PCs;
# the published panel is regenerated by figures/supp_figure_2.py.
if RECOMPUTE_UPSTREAM:
    N_values = [50, 100, 200, 400, 800]
    X_embed_demo = pp.delay_embed(all_pcs, d=7, tau=1)
    Ns, H, Hs = pp.entropy_gap(X_embed_demo[:200000], N_values, lag=lag,
                                framerate=fs, seed=0, n_init=5, verbose=True)
    fig, ax = plt.subplots(figsize=(4.0, 2.8))
    ax.plot(Ns, Hs - H, 'o-')
    ax.set_xscale('log'); ax.set_xlabel('$N$'); ax.set_ylabel(r'$\Delta H(N)$ (nats/s)')
    plt.tight_layout(); fg.save_panel(fig, 'figS2B_entropy_gap'); plt.show()
else:
    print('Fig. S2B (entropy gap) needs the wavelet PCs; see figures/supp_figure_2.py')

In [ ]:
# Fig. S2C: r_k(tau) vs lag for the leading 4 modes.
lags_seconds = np.array([0.1, 0.5, 1.0, 2.0, 3.0, 5.0, 10.0, 30.0, 60.0])
lags_frames = np.unique((lags_seconds * fs).astype(int))
r_k = np.zeros((len(lags_frames), 4))
for li, lg in enumerate(lags_frames):
    Tk = pp.make_transition_matrix(states_list, lag=int(lg), n_states=N)
    ev, _ = pp.leading_eigvecs(Tk, k=4)
    r_k[li] = -np.log(np.abs(ev[:4])) / (lg / fs)
fig, ax = plt.subplots(figsize=(4.0, 2.8))
for k in range(4):
    ax.plot(lags_frames / fs, r_k[:, k], 'o-', label=f'$r_{{{k+2}}}$')
ax.set_xscale('log'); ax.set_yscale('log')
ax.set_xlabel(r'$\tau$ (s)'); ax.set_ylabel(r'$r_k(\tau)$ (1/s)'); ax.legend()
plt.tight_layout()
fg.save_panel(fig, 'figS2C_basin_non_markovian')
plt.show()


## Fig. S3 — per-worm individuality

Refit the M = 2 G-PCCA partition on each worm's own cluster sub-sequence and
match the resulting arm directions to the pooled fit. The high per-worm
cosines show that the run/pirouette geometry is reproducible at the
individual level (Supp. Fig. S3 of the paper also reports the biological
overlays and the leave-one-worm-out cross-validation).

In [ ]:
# Per-worm refit: rebuild T(tau) on each worm's own cluster sequence, rerun
# G-PCCA, and match the resulting arm directions to the pooled fit by best
# 2x2 cosine pairing.  Per-individual sequences visit only a subset of the
# N=250 global clusters, so we restrict T to the visited subset before
# G-PCCA (full-N T has empty rows that pyGPCCA rejects).
arm_dirs_pool = gu.compute_hub_arms(evecs_multi[:, :2].real, pi_multi, chi)['arm_dirs']

cos_per_worm = []
for w in sorted(states_flat):
    states_w = states_flat[w]
    if len(states_w) < lag * 4:
        continue

    visited = np.unique(states_w)
    remap = -np.ones(N, dtype=int); remap[visited] = np.arange(len(visited))
    states_w_r = remap[states_w]
    Tw = pp.make_transition_matrix(states_w_r, lag=lag, n_states=len(visited))
    pi_w = pp.stationary_distribution(Tw)
    try:
        out_w = gu.run_gpcca(Tw, M=2, eta=pi_w)
    except Exception as e:
        print(f'worm {w}: GPCCA failed ({e})'); continue
    ev_w, vc_w = pp.leading_eigvecs(Tw, k=2)
    arms_w = gu.compute_hub_arms(vc_w[:, :2].real, pi_w, out_w['chi'])['arm_dirs']

    c00 = abs(arms_w[0] @ arm_dirs_pool[0]); c11 = abs(arms_w[1] @ arm_dirs_pool[1])
    c01 = abs(arms_w[0] @ arm_dirs_pool[1]); c10 = abs(arms_w[1] @ arm_dirs_pool[0])
    if c00 + c11 >= c01 + c10:
        cos_per_worm.append([c00, c11])
    else:
        cos_per_worm.append([c01, c10])
cos_per_worm = np.array(cos_per_worm)

fig, ax = plt.subplots(figsize=(4.4, 3.0))
ax.boxplot(cos_per_worm, labels=['basin 1', 'basin 2'])
ax.set_ylim(0, 1.05)
ax.set_ylabel('per-worm arm cosine vs pooled fit')
plt.tight_layout()
fg.save_panel(fig, 'figS3_per_worm_arm_cosines')
plt.show()
print('median cosines:', np.median(cos_per_worm, axis=0))

---

All worm panels are saved as PNG/PDF under `outputs/`. Continue to
`flies.ipynb` for the load-bearing fly methodology and biology figures.


## Where to next

- **Read the biology.**  *Results Sec. "C. elegans"* — why the M = 2 partition
  recovers the canonical Pirouette / Run distinction (cf. Stephens 2008,
  Costa 2024).
- **Canonical paper figures.**  `figures/figure_3.py` and
  `figures/supp_figure_{2,3,5,6}.py` regenerate the published panels from the
  cached files in `data/worms/`.
- **Dwell-time tails and the time-evolving landscape.**  Supp. Figs. S5 (model
  selection + slow-mode shape), S6 (the run-well deepening over a recording),
  S10 (Markov-surrogate null), and S11 (smoothing-scale robustness) characterize
  the metastable residence times.
- **Apply this pipeline to your own data.**  Open `user_data.ipynb`.